In [1]:
# Data manipulation and preprocessing
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler

# Classification models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np
# Evaluation metrics
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             precision_score, recall_score, f1_score, make_scorer, roc_curve, accuracy_score)

# Feature selection
from sklearn.feature_selection import SelectKBest, chi2, RFE

# Data balancing (if necessary)
from imblearn.over_sampling import SMOTE

# Handling warnings
import warnings
warnings.filterwarnings("ignore")

/usr/local/lib/python3.10/dist-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [2]:
# Load the dataset

df_filtered = pd.read_csv("combined_df.csv")

In [3]:
# Select relevant symptom columns
symptom_columns = ['Dribbling', 'Swallowing', 'Vomiting', 'Constipation', 'Bowel inconsistence',
                   'Bowel emptying incomplete', 'Urgency', 'Nocturia', 'Pains', 'Weight',
                   'Sweating', 'Diplopia', 'Remembering', 'Loss of interest', 'Concentrating',
                   'Taste/smelling', 'Hallucinations', 'Delusions', 'Sad, blues', 'Anxiety',
                   'Sex drive', 'Sex difficulty', 'Dizzy', 'Falling', 'Swelling',
                   'Daytime sleepiness', 'Insomnia', 'Intense vivid dreams',
                   'Acting out during dreams', 'Restless legs']

KNN, Random Forest, Logistic Regression

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

# Assuming df_filtered and symptom_columns are already defined

# Extract features and target
X = df_filtered[symptom_columns]
y = df_filtered['Rencoded'].apply(lambda x: 0 if x == "Healthy" else (1 if x == "Other_Disorders" else 2))

# 80-20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

# Initialize classifiers to evaluate
classifiers = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
}

# 10-fold cross-validation
kf = StratifiedKFold(n_splits=10)

for model_name, model in classifiers.items():
    print(f"\nEvaluating {model_name}...")

    # For accumulating weighted metrics over the folds
    weighted_precisions, weighted_recalls, weighted_f1s = [], [], []

    # Loop over each fold for cross-validation
    for train_idx, val_idx in kf.split(X_train_balanced, y_train_balanced):
        # Split the data for this fold
        X_fold_train, X_fold_val = X_train_balanced[train_idx], X_train_balanced[val_idx]
        y_fold_train, y_fold_val = y_train_balanced[train_idx], y_train_balanced[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_val_pred = model.predict(X_fold_val)

        # Calculate metrics for the current fold
        fold_precision, fold_recall, fold_f1, _ = precision_recall_fscore_support(
            y_fold_val, y_val_pred, average='weighted'
        )

        # Accumulate metrics
        weighted_precisions.append(fold_precision)
        weighted_recalls.append(fold_recall)
        weighted_f1s.append(fold_f1)

    # Calculate average weighted metrics across all folds
    avg_weighted_precision = np.mean(weighted_precisions)
    avg_weighted_recall = np.mean(weighted_recalls)
    avg_weighted_f1 = np.mean(weighted_f1s)

    print(f"\nWeighted Average (Training) for {model_name}:")
    print(f"Precision: {avg_weighted_precision:.2f}, Recall: {avg_weighted_recall:.2f}, F1-Score: {avg_weighted_f1:.2f}")

    # Evaluate on the hold-out test set
    model.fit(X_train_balanced, y_train_balanced)
    y_test_pred = model.predict(X_test_scaled)

    # Classification report and confusion matrix for the test set
    test_report = classification_report(y_test, y_test_pred, target_names=["Healthy", "Other_Disorders", "Parkinson's"])
    print("\nTest Set Evaluation:")
    print(test_report)

    # Confusion matrix for the test set
    test_conf_matrix = confusion_matrix(y_test, y_test_pred)
    print(f"Confusion Matrix for {model_name} (Testing):\n{test_conf_matrix}")



Evaluating KNN...

Weighted Average (Training) for KNN:
Precision: 0.76, Recall: 0.73, F1-Score: 0.72

Test Set Evaluation:
                 precision    recall  f1-score   support

        Healthy       0.37      0.81      0.51        16
Other_Disorders       0.38      0.43      0.41        23
    Parkinson's       0.70      0.42      0.52        55

       accuracy                           0.49        94
      macro avg       0.48      0.56      0.48        94
   weighted avg       0.57      0.49      0.49        94

Confusion Matrix for KNN (Testing):
[[13  0  3]
 [ 6 10  7]
 [16 16 23]]

Evaluating Random Forest...

Weighted Average (Training) for Random Forest:
Precision: 0.82, Recall: 0.81, F1-Score: 0.80

Test Set Evaluation:
                 precision    recall  f1-score   support

        Healthy       0.62      0.62      0.62        16
Other_Disorders       0.54      0.30      0.39        23
    Parkinson's       0.71      0.84      0.77        55

       accuracy          

Gradient Boosting, AdaBoost, Naive Bayes

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
import numpy as np
import pandas as pd


# Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

# Initialize classifiers to evaluate
classifiers = {
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Naive Bayes': GaussianNB()
}

# 10-fold cross-validation
kf = StratifiedKFold(n_splits=10)

for model_name, model in classifiers.items():
    print(f"\nEvaluating {model_name}...")

    # For accumulating weighted metrics over the folds
    weighted_precisions, weighted_recalls, weighted_f1s = [], [], []

    # Loop over each fold for cross-validation
    for train_idx, val_idx in kf.split(X_train_balanced, y_train_balanced):
        # Split the data for this fold
        X_fold_train, X_fold_val = X_train_balanced[train_idx], X_train_balanced[val_idx]
        y_fold_train, y_fold_val = y_train_balanced[train_idx], y_train_balanced[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_val_pred = model.predict(X_fold_val)

        # Calculate metrics for the current fold
        fold_precision, fold_recall, fold_f1, _ = precision_recall_fscore_support(
            y_fold_val, y_val_pred, average='weighted'
        )

        # Accumulate metrics
        weighted_precisions.append(fold_precision)
        weighted_recalls.append(fold_recall)
        weighted_f1s.append(fold_f1)

    # Calculate average weighted metrics across all folds
    avg_weighted_precision = np.mean(weighted_precisions)
    avg_weighted_recall = np.mean(weighted_recalls)
    avg_weighted_f1 = np.mean(weighted_f1s)

    print(f"\nWeighted Average (Training) for {model_name}:")
    print(f"Precision: {avg_weighted_precision:.2f}, Recall: {avg_weighted_recall:.2f}, F1-Score: {avg_weighted_f1:.2f}")

    # Evaluate on the hold-out test set
    model.fit(X_train_balanced, y_train_balanced)
    y_test_pred = model.predict(X_test_scaled)

    # Classification report and confusion matrix for the test set
    test_report = classification_report(y_test, y_test_pred, target_names=["Healthy", "Other_Disorders", "Parkinson's"])
    print("\nTest Set Evaluation:")
    print(test_report)

    # Confusion matrix for the test set
    test_conf_matrix = confusion_matrix(y_test, y_test_pred)
    print(f"Confusion Matrix for {model_name} (Testing):\n{test_conf_matrix}")



Evaluating Gradient Boosting...

Weighted Average (Training) for Gradient Boosting:
Precision: 0.80, Recall: 0.79, F1-Score: 0.78

Test Set Evaluation:
                 precision    recall  f1-score   support

        Healthy       0.57      0.81      0.67        16
Other_Disorders       0.50      0.39      0.44        23
    Parkinson's       0.75      0.73      0.74        55

       accuracy                           0.66        94
      macro avg       0.61      0.64      0.62        94
   weighted avg       0.66      0.66      0.65        94

Confusion Matrix for Gradient Boosting (Testing):
[[13  0  3]
 [ 4  9 10]
 [ 6  9 40]]

Evaluating AdaBoost...

Weighted Average (Training) for AdaBoost:
Precision: 0.67, Recall: 0.63, F1-Score: 0.61

Test Set Evaluation:
                 precision    recall  f1-score   support

        Healthy       0.59      0.62      0.61        16
Other_Disorders       0.28      0.48      0.35        23
    Parkinson's       0.66      0.45      0.54     

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

# Assuming df_filtered and symptom_columns are already defined

# Extract features and target
X = df_filtered[symptom_columns]
y = df_filtered['Rencoded'].apply(lambda x: 0 if x == "Healthy" else (1 if x == "Other_Disorders" else 2))

# 80-20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

# Initialize classifiers to evaluate
classifiers = {
    "Decision Tree": DecisionTreeClassifier(),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    "LightGBM": lgb.LGBMClassifier()
}

# 10-fold cross-validation
kf = StratifiedKFold(n_splits=10)

for model_name, model in classifiers.items():
    print(f"\nEvaluating {model_name}...")

    # For accumulating weighted metrics over the folds
    weighted_precisions, weighted_recalls, weighted_f1s = [], [], []

    # Loop over each fold for cross-validation
    for train_idx, val_idx in kf.split(X_train_balanced, y_train_balanced):
        # Split the data for this fold
        X_fold_train, X_fold_val = X_train_balanced[train_idx], X_train_balanced[val_idx]
        y_fold_train, y_fold_val = y_train_balanced[train_idx], y_train_balanced[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_val_pred = model.predict(X_fold_val)

        # Calculate metrics for the current fold
        fold_precision, fold_recall, fold_f1, _ = precision_recall_fscore_support(
            y_fold_val, y_val_pred, average='weighted'
        )

        # Accumulate metrics
        weighted_precisions.append(fold_precision)
        weighted_recalls.append(fold_recall)
        weighted_f1s.append(fold_f1)

    # Calculate average weighted metrics across all folds
    avg_weighted_precision = np.mean(weighted_precisions)
    avg_weighted_recall = np.mean(weighted_recalls)
    avg_weighted_f1 = np.mean(weighted_f1s)

    print(f"\nWeighted Average (Training) for {model_name}:")
    print(f"Precision: {avg_weighted_precision:.2f}, Recall: {avg_weighted_recall:.2f}, F1-Score: {avg_weighted_f1:.2f}")

    # Evaluate on the hold-out test set
    model.fit(X_train_balanced, y_train_balanced)
    y_test_pred = model.predict(X_test_scaled)

    # Classification report and confusion matrix for the test set
    test_report = classification_report(y_test, y_test_pred, target_names=["Healthy", "Other_Disorders", "Parkinson's"])
    print("\nTest Set Evaluation:")
    print(test_report)

    # Confusion matrix for the test set
    test_conf_matrix = confusion_matrix(y_test, y_test_pred)
    print(f"Confusion Matrix for {model_name} (Testing):\n{test_conf_matrix}")


Evaluating Decision Tree...

Weighted Average (Training) for Decision Tree:
Precision: 0.76, Recall: 0.75, F1-Score: 0.74

Test Set Evaluation:
                 precision    recall  f1-score   support

        Healthy       0.47      0.56      0.51        16
Other_Disorders       0.26      0.22      0.24        23
    Parkinson's       0.64      0.65      0.65        55

       accuracy                           0.53        94
      macro avg       0.46      0.48      0.47        94
   weighted avg       0.52      0.53      0.53        94

Confusion Matrix for Decision Tree (Testing):
[[ 9  1  6]
 [ 4  5 14]
 [ 6 13 36]]

Evaluating XGB...

Weighted Average (Training) for XGB:
Precision: 0.80, Recall: 0.78, F1-Score: 0.77

Test Set Evaluation:
                 precision    recall  f1-score   support

        Healthy       0.50      0.81      0.62        16
Other_Disorders       0.62      0.43      0.51        23
    Parkinson's       0.79      0.75      0.77        55

       accuracy

In [4]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 7.9 MB/s eta 0:00:00


Svm, mlp, polynomial, cat

In [5]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

# Assuming df_filtered and symptom_columns are already defined

# Extract features and target
X = df_filtered[symptom_columns]
y = df_filtered['Rencoded'].apply(lambda x: 0 if x == "Healthy" else (1 if x == "Other_Disorders" else 2))

# 80-20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

# Initialize classifiers to evaluate
classifiers = {
   'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}

# 10-fold cross-validation
kf = StratifiedKFold(n_splits=10)

for model_name, model in classifiers.items():
    print(f"\nEvaluating {model_name}...")

    # For accumulating weighted metrics over the folds
    weighted_precisions, weighted_recalls, weighted_f1s = [], [], []

    # Loop over each fold for cross-validation
    for train_idx, val_idx in kf.split(X_train_balanced, y_train_balanced):
        # Split the data for this fold
        X_fold_train, X_fold_val = X_train_balanced[train_idx], X_train_balanced[val_idx]
        y_fold_train, y_fold_val = y_train_balanced[train_idx], y_train_balanced[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_val_pred = model.predict(X_fold_val)

        # Calculate metrics for the current fold
        fold_precision, fold_recall, fold_f1, _ = precision_recall_fscore_support(
            y_fold_val, y_val_pred, average='weighted'
        )

        # Accumulate metrics
        weighted_precisions.append(fold_precision)
        weighted_recalls.append(fold_recall)
        weighted_f1s.append(fold_f1)

    # Calculate average weighted metrics across all folds
    avg_weighted_precision = np.mean(weighted_precisions)
    avg_weighted_recall = np.mean(weighted_recalls)
    avg_weighted_f1 = np.mean(weighted_f1s)

    print(f"\nWeighted Average (Training) for {model_name}:")
    print(f"Precision: {avg_weighted_precision:.2f}, Recall: {avg_weighted_recall:.2f}, F1-Score: {avg_weighted_f1:.2f}")

    # Evaluate on the hold-out test set
    model.fit(X_train_balanced, y_train_balanced)
    y_test_pred = model.predict(X_test_scaled)

    # Classification report and confusion matrix for the test set
    test_report = classification_report(y_test, y_test_pred, target_names=["Healthy", "Other_Disorders", "Parkinson's"])
    print("\nTest Set Evaluation:")
    print(test_report)

    # Confusion matrix for the test set
    test_conf_matrix = confusion_matrix(y_test, y_test_pred)
    print(f"Confusion Matrix for {model_name} (Testing):\n{test_conf_matrix}")


Evaluating SVM...

Weighted Average (Training) for SVM:
Precision: 0.84, Recall: 0.83, F1-Score: 0.83

Test Set Evaluation:
                 precision    recall  f1-score   support

        Healthy       0.46      0.75      0.57        16
Other_Disorders       0.44      0.17      0.25        23
    Parkinson's       0.71      0.76      0.74        55

       accuracy                           0.62        94
      macro avg       0.54      0.56      0.52        94
   weighted avg       0.60      0.62      0.59        94

Confusion Matrix for SVM (Testing):
[[12  0  4]
 [ 6  4 13]
 [ 8  5 42]]

Evaluating MLP...

Weighted Average (Training) for MLP:
Precision: 0.87, Recall: 0.86, F1-Score: 0.86

Test Set Evaluation:
                 precision    recall  f1-score   support

        Healthy       0.50      0.50      0.50        16
Other_Disorders       0.32      0.26      0.29        23
    Parkinson's       0.69      0.75      0.72        55

       accuracy                           0.5

With Smote

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import numpy as np
import pandas as pd

# Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

# Initialize classifiers to evaluate
classifiers = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'XGB': XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, use_label_encoder=False, eval_metric="mlogloss"),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
    'LightGBM': LGBMClassifier(random_state=42)
}

# 10-fold cross-validation
kf = StratifiedKFold(n_splits=10)

for model_name, model in classifiers.items():
    print(f"\nEvaluating {model_name}...")

    # For accumulating weighted metrics over the folds
    weighted_precisions, weighted_recalls, weighted_f1s = [], [], []

    # Loop over each fold for cross-validation
    for train_idx, val_idx in kf.split(X_train_balanced, y_train_balanced):
        # Split the data for this fold
        X_fold_train, X_fold_val = X_train_balanced[train_idx], X_train_balanced[val_idx]
        y_fold_train, y_fold_val = y_train_balanced[train_idx], y_train_balanced[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_val_pred = model.predict(X_fold_val)

        # Calculate metrics for the current fold
        fold_precision, fold_recall, fold_f1, _ = precision_recall_fscore_support(
            y_fold_val, y_val_pred, average='weighted'
        )

        # Accumulate metrics
        weighted_precisions.append(fold_precision)
        weighted_recalls.append(fold_recall)
        weighted_f1s.append(fold_f1)

    # Calculate average weighted metrics across all folds
    avg_weighted_precision = np.mean(weighted_precisions)
    avg_weighted_recall = np.mean(weighted_recalls)
    avg_weighted_f1 = np.mean(weighted_f1s)

    print(f"\nWeighted Average (Training) for {model_name}:")
    print(f"Precision: {avg_weighted_precision:.2f}, Recall: {avg_weighted_recall:.2f}, F1-Score: {avg_weighted_f1:.2f}")

    # Evaluate on the hold-out test set
    model.fit(X_train_balanced, y_train_balanced)
    y_test_pred = model.predict(X_test_scaled)

    # Classification report and confusion matrix for the test set
    test_report = classification_report(y_test, y_test_pred, target_names=["Healthy", "Other_Disorders", "Parkinson's"])
    print("\nTest Set Evaluation:")
    print(test_report)

    # Confusion matrix for the test set
    test_conf_matrix = confusion_matrix(y_test, y_test_pred)
    print(f"Confusion Matrix for {model_name} (Testing):\n{test_conf_matrix}")



Evaluating KNN...

Weighted Average (Training) for KNN:
Precision: 0.76, Recall: 0.73, F1-Score: 0.72

Test Set Evaluation:
                 precision    recall  f1-score   support

        Healthy       0.37      0.81      0.51        16
Other_Disorders       0.38      0.43      0.41        23
    Parkinson's       0.70      0.42      0.52        55

       accuracy                           0.49        94
      macro avg       0.48      0.56      0.48        94
   weighted avg       0.57      0.49      0.49        94

Confusion Matrix for KNN (Testing):
[[13  0  3]
 [ 6 10  7]
 [16 16 23]]

Evaluating Random Forest...

Weighted Average (Training) for Random Forest:
Precision: 0.82, Recall: 0.81, F1-Score: 0.80

Test Set Evaluation:
                 precision    recall  f1-score   support

        Healthy       0.62      0.62      0.62        16
Other_Disorders       0.54      0.30      0.39        23
    Parkinson's       0.71      0.84      0.77        55

       accuracy          

SVM, MLP, cat and polynomial with smote

In [6]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import numpy as np
import pandas as pd

# Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

# Initialize classifiers to evaluate
classifiers = {
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}

# 10-fold cross-validation
kf = StratifiedKFold(n_splits=10)

for model_name, model in classifiers.items():
    print(f"\nEvaluating {model_name}...")

    # For accumulating weighted metrics over the folds
    weighted_precisions, weighted_recalls, weighted_f1s = [], [], []

    # Loop over each fold for cross-validation
    for train_idx, val_idx in kf.split(X_train_balanced, y_train_balanced):
        # Split the data for this fold
        X_fold_train, X_fold_val = X_train_balanced[train_idx], X_train_balanced[val_idx]
        y_fold_train, y_fold_val = y_train_balanced[train_idx], y_train_balanced[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_val_pred = model.predict(X_fold_val)

        # Calculate metrics for the current fold
        fold_precision, fold_recall, fold_f1, _ = precision_recall_fscore_support(
            y_fold_val, y_val_pred, average='weighted'
        )

        # Accumulate metrics
        weighted_precisions.append(fold_precision)
        weighted_recalls.append(fold_recall)
        weighted_f1s.append(fold_f1)

    # Calculate average weighted metrics across all folds
    avg_weighted_precision = np.mean(weighted_precisions)
    avg_weighted_recall = np.mean(weighted_recalls)
    avg_weighted_f1 = np.mean(weighted_f1s)

    print(f"\nWeighted Average (Training) for {model_name}:")
    print(f"Precision: {avg_weighted_precision:.2f}, Recall: {avg_weighted_recall:.2f}, F1-Score: {avg_weighted_f1:.2f}")

    # Evaluate on the hold-out test set
    model.fit(X_train_balanced, y_train_balanced)
    y_test_pred = model.predict(X_test_scaled)

    # Classification report and confusion matrix for the test set
    test_report = classification_report(y_test, y_test_pred, target_names=["Healthy", "Other_Disorders", "Parkinson's"])
    print("\nTest Set Evaluation:")
    print(test_report)

    # Confusion matrix for the test set
    test_conf_matrix = confusion_matrix(y_test, y_test_pred)
    print(f"Confusion Matrix for {model_name} (Testing):\n{test_conf_matrix}")


Evaluating SVM...

Weighted Average (Training) for SVM:
Precision: 0.84, Recall: 0.83, F1-Score: 0.83

Test Set Evaluation:
                 precision    recall  f1-score   support

        Healthy       0.46      0.75      0.57        16
Other_Disorders       0.44      0.17      0.25        23
    Parkinson's       0.71      0.76      0.74        55

       accuracy                           0.62        94
      macro avg       0.54      0.56      0.52        94
   weighted avg       0.60      0.62      0.59        94

Confusion Matrix for SVM (Testing):
[[12  0  4]
 [ 6  4 13]
 [ 8  5 42]]

Evaluating MLP...

Weighted Average (Training) for MLP:
Precision: 0.87, Recall: 0.86, F1-Score: 0.86

Test Set Evaluation:
                 precision    recall  f1-score   support

        Healthy       0.50      0.50      0.50        16
Other_Disorders       0.32      0.26      0.29        23
    Parkinson's       0.69      0.75      0.72        55

       accuracy                           0.5

smote 0.5

In [11]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

# Example: Split your dataset into X_train_scaled and y_train (replace with your actual data)
# X_train_scaled = <your feature data>
# y_train = <your target data>
# X_test_scaled = <your test feature data>
# y_test = <your test target data>

# Get class distribution in training data
class_counts = y_train.value_counts()

# Determine the majority class (with the most samples)
majority_class_count = class_counts.max()

# Create a dictionary for the sampling strategy where:
# 1. We apply 50% oversampling for minority classes.
# 2. The majority class count is kept unchanged.
sampling_strategy = {class_label: max(int(majority_class_count * 0.5), class_counts[class_label])
                     for class_label in class_counts.index}

# Apply SMOTE with the manually calculated sampling strategy
smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42)

# Apply SMOTE to balance the training data
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

# Initialize classifiers to evaluate
classifiers = {
    'KNN': KNeighborsClassifier(),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'XGB': XGBClassifier(use_label_encoder=False, eval_metric="mlogloss"),
    'Extra Trees': ExtraTreesClassifier(random_state=42),
    'LightGBM': LGBMClassifier(random_state=42),
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}

# 10-fold cross-validation
kf = StratifiedKFold(n_splits=10)

for model_name, model in classifiers.items():
    print(f"\nEvaluating {model_name}...")

    # For accumulating weighted metrics over the folds
    weighted_precisions, weighted_recalls, weighted_f1s = [], [], []

    # Loop over each fold for cross-validation
    for train_idx, val_idx in kf.split(X_train_balanced, y_train_balanced):
        # Split the data for this fold
        X_fold_train, X_fold_val = X_train_balanced[train_idx], X_train_balanced[val_idx]
        y_fold_train, y_fold_val = y_train_balanced[train_idx], y_train_balanced[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_val_pred = model.predict(X_fold_val)

        # Calculate metrics for the current fold
        fold_precision, fold_recall, fold_f1, _ = precision_recall_fscore_support(
            y_fold_val, y_val_pred, average='weighted'
        )

        # Accumulate metrics
        weighted_precisions.append(fold_precision)
        weighted_recalls.append(fold_recall)
        weighted_f1s.append(fold_f1)

    # Calculate average weighted metrics across all folds
    avg_weighted_precision = np.mean(weighted_precisions)
    avg_weighted_recall = np.mean(weighted_recalls)
    avg_weighted_f1 = np.mean(weighted_f1s)

    print(f"\nWeighted Average (Training) for {model_name}:")
    print(f"Precision: {avg_weighted_precision:.2f}, Recall: {avg_weighted_recall:.2f}, F1-Score: {avg_weighted_f1:.2f}")

    # Evaluate on the hold-out test set
    model.fit(X_train_balanced, y_train_balanced)
    y_test_pred = model.predict(X_test_scaled)

    # Classification report and confusion matrix for the test set
    test_report = classification_report(y_test, y_test_pred, target_names=["Healthy", "Other_Disorders", "Parkinson's"])
    print("\nTest Set Evaluation:")
    print(test_report)

    # Confusion matrix for the test set
    test_conf_matrix = confusion_matrix(y_test, y_test_pred)
    print(f"Confusion Matrix for {model_name} (Testing):\n{test_conf_matrix}")



Evaluating KNN...

Weighted Average (Training) for KNN:
Precision: 0.61, Recall: 0.59, F1-Score: 0.57

Test Set Evaluation:
                 precision    recall  f1-score   support

        Healthy       0.37      0.81      0.51        16
Other_Disorders       0.17      0.09      0.11        23
    Parkinson's       0.70      0.60      0.65        55

       accuracy                           0.51        94
      macro avg       0.41      0.50      0.42        94
   weighted avg       0.51      0.51      0.49        94

Confusion Matrix for KNN (Testing):
[[13  1  2]
 [ 9  2 12]
 [13  9 33]]

Evaluating Random Forest...

Weighted Average (Training) for Random Forest:
Precision: 0.70, Recall: 0.72, F1-Score: 0.68

Test Set Evaluation:
                 precision    recall  f1-score   support

        Healthy       0.60      0.56      0.58        16
Other_Disorders       0.50      0.22      0.30        23
    Parkinson's       0.70      0.87      0.77        55

       accuracy          

Hyperparameter

In [21]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

# Replace these placeholders with your actual data
# X_train_scaled, y_train, X_test_scaled, y_test = ...


# Apply SMOTE for class balancing
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

# Define hyperparameter grids for each classifier
param_grids = {
    'KNN': {'n_neighbors': [3, 5, 7], 'weights': ['uniform', 'distance']},
    'Random Forest': {'n_estimators': [50, 100, 150], 'max_depth': [None, 10, 20]},
    'Logistic Regression': {'C': [0.1, 1, 10], 'penalty': ['l2'], 'solver': ['lbfgs']},
    'Gradient Boosting': {'n_estimators': [50, 100, 150], 'learning_rate': [0.01, 0.1, 0.2]},
    'AdaBoost': {'n_estimators': [50, 100, 150], 'learning_rate': [0.01, 0.1, 0.2]},
    'Naive Bayes': {},  # No hyperparameters to tune
    'Decision Tree': {'max_depth': [None, 10, 20], 'min_samples_split': [2, 5, 10]},
    'XGB': {'n_estimators': [50, 100, 150], 'learning_rate': [0.01, 0.1, 0.2], 'max_depth': [3, 5, 7]},
    'Extra Trees': {'n_estimators': [50, 100, 150], 'max_depth': [None, 10, 20]},
    'LightGBM': {'n_estimators': [50, 100, 150], 'learning_rate': [0.01, 0.1, 0.2]},
    'SVM': {'C': [0.1, 1, 10], 'kernel': ['linear', 'poly', 'rbf'], 'gamma': ['scale', 'auto']},
    'MLP': {'hidden_layer_sizes': [(50,), (100,), (50, 50)], 'activation': ['relu', 'tanh', 'logistic']},
    'CatBoost': {'iterations': [100, 200], 'learning_rate': [0.01, 0.1], 'depth': [3, 5]},
    'Polynomial Regression': {'poly__degree': [2, 3], 'model__C': [0.1, 1, 10]}
}

# Define classifiers
classifiers = {
    'KNN': KNeighborsClassifier(),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'XGB': XGBClassifier(use_label_encoder=False, eval_metric="mlogloss"),
    'Extra Trees': ExtraTreesClassifier(random_state=42),
    'LightGBM': LGBMClassifier(random_state=42),
    'SVM': SVC(random_state=42),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures()), ('model', LogisticRegression(max_iter=1000))])
}

# Stratified k-fold cross-validation setup
kf = StratifiedKFold(n_splits=10)

# Evaluate each model
for model_name, model in classifiers.items():
    print(f"\nEvaluating {model_name} with Hyperparameter Tuning...")

    # Set up GridSearchCV with the parameter grid for the current model
    grid_search = GridSearchCV(model, param_grids.get(model_name, {}), cv=3, scoring='f1_weighted', n_jobs=-1)
    grid_search.fit(X_train_balanced, y_train_balanced)
    best_model = grid_search.best_estimator_

    # Display the best hyperparameters
    print(f"Best Parameters for {model_name}: {grid_search.best_params_}")

    # Perform cross-validation with the best model
    weighted_precisions, weighted_recalls, weighted_f1s = [], [], []
    for train_idx, val_idx in kf.split(X_train_balanced, y_train_balanced):
        X_fold_train, X_fold_val = X_train_balanced[train_idx], X_train_balanced[val_idx]
        y_fold_train, y_fold_val = y_train_balanced[train_idx], y_train_balanced[val_idx]

        # Train and evaluate the model
        best_model.fit(X_fold_train, y_fold_train)
        y_val_pred = best_model.predict(X_fold_val)
        fold_precision, fold_recall, fold_f1, _ = precision_recall_fscore_support(
            y_fold_val, y_val_pred, average='weighted'
        )
        weighted_precisions.append(fold_precision)
        weighted_recalls.append(fold_recall)
        weighted_f1s.append(fold_f1)

    # Calculate and display average weighted metrics across all folds
    avg_precision = np.mean(weighted_precisions)
    avg_recall = np.mean(weighted_recalls)
    avg_f1 = np.mean(weighted_f1s)
    print(f"\nCross-Validation Metrics for {model_name} (Weighted):")
    print(f"Precision: {avg_precision:.2f}, Recall: {avg_recall:.2f}, F1-Score: {avg_f1:.2f}")

    # Evaluate the tuned model on the test set
    best_model.fit(X_train_balanced, y_train_balanced)
    y_test_pred = best_model.predict(X_test_scaled)
    print(f"\nTest Set Evaluation for {model_name}:")
    print(classification_report(y_test, y_test_pred, target_names=["Healthy", "Other_Disorders", "Parkinson's"]))
    print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_test_pred)}")



Evaluating KNN with Hyperparameter Tuning...
Best Parameters for KNN: {'n_neighbors': 5, 'weights': 'distance'}

Cross-Validation Metrics for KNN (Weighted):
Precision: 0.76, Recall: 0.72, F1-Score: 0.71

Test Set Evaluation for KNN:
                 precision    recall  f1-score   support

        Healthy       0.40      0.75      0.52        16
Other_Disorders       0.39      0.52      0.44        23
    Parkinson's       0.73      0.44      0.55        55

       accuracy                           0.51        94
      macro avg       0.50      0.57      0.50        94
   weighted avg       0.59      0.51      0.52        94

Confusion Matrix:
[[12  1  3]
 [ 5 12  6]
 [13 18 24]]

Evaluating Random Forest with Hyperparameter Tuning...
Best Parameters for Random Forest: {'max_depth': None, 'n_estimators': 150}

Cross-Validation Metrics for Random Forest (Weighted):
Precision: 0.82, Recall: 0.81, F1-Score: 0.80

Test Set Evaluation for Random Forest:
                 precision    reca

hyperparameter with smote 0.5

In [22]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

# Replace these placeholders with your actual data
# X_train_scaled, y_train, X_test_scaled, y_test = ...

# Determine the sampling strategy for SMOTE
class_counts = y_train.value_counts()
majority_class_count = class_counts.max()
sampling_strategy = {class_label: max(int(majority_class_count * 0.5), class_counts[class_label])
                     for class_label in class_counts.index}

# Apply SMOTE for class balancing
smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

# Define hyperparameter grids for each classifier
param_grids = {
    'KNN': {'n_neighbors': [3, 5, 7], 'weights': ['uniform', 'distance']},
    'Random Forest': {'n_estimators': [50, 100, 150], 'max_depth': [None, 10, 20]},
    'Logistic Regression': {'C': [0.1, 1, 10], 'penalty': ['l2'], 'solver': ['lbfgs']},
    'Gradient Boosting': {'n_estimators': [50, 100, 150], 'learning_rate': [0.01, 0.1, 0.2]},
    'AdaBoost': {'n_estimators': [50, 100, 150], 'learning_rate': [0.01, 0.1, 0.2]},
    'Naive Bayes': {},  # No hyperparameters to tune
    'Decision Tree': {'max_depth': [None, 10, 20], 'min_samples_split': [2, 5, 10]},
    'XGB': {'n_estimators': [50, 100, 150], 'learning_rate': [0.01, 0.1, 0.2], 'max_depth': [3, 5, 7]},
    'Extra Trees': {'n_estimators': [50, 100, 150], 'max_depth': [None, 10, 20]},
    'LightGBM': {'n_estimators': [50, 100, 150], 'learning_rate': [0.01, 0.1, 0.2]},
    'SVM': {'C': [0.1, 1, 10], 'kernel': ['linear', 'poly', 'rbf'], 'gamma': ['scale', 'auto']},
    'MLP': {'hidden_layer_sizes': [(50,), (100,), (50, 50)], 'activation': ['relu', 'tanh', 'logistic']},
    'CatBoost': {'iterations': [100, 200], 'learning_rate': [0.01, 0.1], 'depth': [3, 5]},
    'Polynomial Regression': {'poly__degree': [2, 3], 'model__C': [0.1, 1, 10]}
}

# Define classifiers
classifiers = {
    'KNN': KNeighborsClassifier(),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'XGB': XGBClassifier(use_label_encoder=False, eval_metric="mlogloss"),
    'Extra Trees': ExtraTreesClassifier(random_state=42),
    'LightGBM': LGBMClassifier(random_state=42),
    'SVM': SVC(random_state=42),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures()), ('model', LogisticRegression(max_iter=1000))])
}

# Stratified k-fold cross-validation setup
kf = StratifiedKFold(n_splits=10)

# Evaluate each model
for model_name, model in classifiers.items():
    print(f"\nEvaluating {model_name} with Hyperparameter Tuning...")

    # Set up GridSearchCV with the parameter grid for the current model
    grid_search = GridSearchCV(model, param_grids.get(model_name, {}), cv=3, scoring='f1_weighted', n_jobs=-1)
    grid_search.fit(X_train_balanced, y_train_balanced)
    best_model = grid_search.best_estimator_

    # Display the best hyperparameters
    print(f"Best Parameters for {model_name}: {grid_search.best_params_}")

    # Perform cross-validation with the best model
    weighted_precisions, weighted_recalls, weighted_f1s = [], [], []
    for train_idx, val_idx in kf.split(X_train_balanced, y_train_balanced):
        X_fold_train, X_fold_val = X_train_balanced[train_idx], X_train_balanced[val_idx]
        y_fold_train, y_fold_val = y_train_balanced[train_idx], y_train_balanced[val_idx]

        # Train and evaluate the model
        best_model.fit(X_fold_train, y_fold_train)
        y_val_pred = best_model.predict(X_fold_val)
        fold_precision, fold_recall, fold_f1, _ = precision_recall_fscore_support(
            y_fold_val, y_val_pred, average='weighted'
        )
        weighted_precisions.append(fold_precision)
        weighted_recalls.append(fold_recall)
        weighted_f1s.append(fold_f1)

    # Calculate and display average weighted metrics across all folds
    avg_precision = np.mean(weighted_precisions)
    avg_recall = np.mean(weighted_recalls)
    avg_f1 = np.mean(weighted_f1s)
    print(f"\nCross-Validation Metrics for {model_name} (Weighted):")
    print(f"Precision: {avg_precision:.2f}, Recall: {avg_recall:.2f}, F1-Score: {avg_f1:.2f}")

    # Evaluate the tuned model on the test set
    best_model.fit(X_train_balanced, y_train_balanced)
    y_test_pred = best_model.predict(X_test_scaled)
    print(f"\nTest Set Evaluation for {model_name}:")
    print(classification_report(y_test, y_test_pred, target_names=["Healthy", "Other_Disorders", "Parkinson's"]))
    print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_test_pred)}")



Evaluating KNN with Hyperparameter Tuning...
Best Parameters for KNN: {'n_neighbors': 3, 'weights': 'distance'}

Cross-Validation Metrics for KNN (Weighted):
Precision: 0.64, Recall: 0.63, F1-Score: 0.62

Test Set Evaluation for KNN:
                 precision    recall  f1-score   support

        Healthy       0.41      0.81      0.54        16
Other_Disorders       0.21      0.13      0.16        23
    Parkinson's       0.62      0.55      0.58        55

       accuracy                           0.49        94
      macro avg       0.42      0.50      0.43        94
   weighted avg       0.49      0.49      0.47        94

Confusion Matrix:
[[13  0  3]
 [ 5  3 15]
 [14 11 30]]

Evaluating Random Forest with Hyperparameter Tuning...
Best Parameters for Random Forest: {'max_depth': None, 'n_estimators': 150}

Cross-Validation Metrics for Random Forest (Weighted):
Precision: 0.69, Recall: 0.71, F1-Score: 0.68

Test Set Evaluation for Random Forest:
                 precision    reca